## **JASPAR 2024 CORE - PWM to Consensus Sequence**

In [31]:
# Libraries
import os

In [32]:
# Paths
jaspar_2026="/scratch/prj/stem_cells_pituitary/Georgia/genome/JASPAR_CORE_2026_non-redundant.meme"
output_tsv="/scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv"
nucleotides = ['A', 'C', 'G', 'T']

os.path.exists(jaspar_2026)

True

In [37]:
# Function for generating .tsv file with no thresholds on sequence

print("Begin parsing MEME file.")

with open(jaspar_2026, "r") as infile, open(output_tsv, "w") as outfile:
    current_id = None
    matrix_lines = []
    is_reading_matrix = False

    for line in infile:
        line = line.strip()
        
        # Identify the start of a motif
        if line.startswith("MOTIF"):
            # End sequence of previous motif
            if current_id and matrix_lines:
                consensus = ""
                for row in matrix_lines:
                    probs = [float(x) for x in row.split()]
                    max_idx = probs.index(max(probs))
                    consensus += nucleotides[max_idx]
                outfile.write(f"{current_id}\t{consensus}\n")
            
            # Begin sequence of new motif
            parts = line.split()
            current_id = parts[1]
            matrix_lines = []
            is_reading_matrix = False
            
        # Identify the letter-probability matrix within the motif information
        elif line.startswith("letter-probability matrix"):
            is_reading_matrix = True
            continue
            
        # Extract the letter-probability matrix
        elif is_reading_matrix:
            if not line or line.startswith("URL"):
                is_reading_matrix = False
                continue
            matrix_lines.append(line)

    # Last motif in the file
    if current_id and matrix_lines:
        consensus = ""
        for row in matrix_lines:
            probs = [float(x) for x in row.split()]
            max_idx = probs.index(max(probs))
            consensus += nucleotides[max_idx]
        outfile.write(f"{current_id}\t{consensus}\n")

print(f"Successfully created {output_tsv}")

Begin parsing MEME file.
Successfully created /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv


In [36]:
!head -n 5 /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv

!grep "MA0784.2" /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv
!grep "MA0112.3" /scratch/prj/stem_cells_pituitary/Georgia/ChromBPnet/data/motif_sequences.tsv

MA0002.1	TATTGTGGTTA
MA0003.1	GCCCGGGGG
MA0004.1	CACGTG
MA0006.1	TGCGTG
MA0007.1	ATAAGAACACCCTGTACCCGCC
MA0784.2	AGCTAATTTGCATAAT
MA0112.3	AAGGTCACGGTGACCTG


In [ ]:
# Function for generating .tsv file with added thresholds on sequence
# e.g., adding letter N if confused

